In [15]:
import time
from pyspark.sql import functions as F
from pyspark.sql.types import *
from spark_session import get_spark

spark = get_spark(
    "FastFood-P6-Pipeline",
    extra_packages=["org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1",
                    "io.delta:delta-spark_2.12:3.2.0"],
    extra_conf={"spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension",
                "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"})
spark.sparkContext.setLogLevel("WARN")

LAKE   = "s3a://lake"
SILVER = f"{LAKE}/silver/orders_clean"
GOLD   = f"{LAKE}/gold/kpi_city_10min"
CKPT   = f"{LAKE}/checkpoints"

In [18]:
   df = spark.read.parquet("s3a://lake/silver/orders_clean")
   print("Silver (lignes, couples distincts) :", df.count(), df.select("order_id", "status").distinct().count())

Silver (lignes, couples distincts) : 15737 15737


In [19]:
orders_schema = StructType([
    StructField("order_id", StringType()), StructField("restaurant_id", StringType()),
    StructField("customer_id", StringType()), StructField("city", StringType()),
    StructField("status", StringType()), StructField("amount_mad", DoubleType()),
    StructField("items_count", IntegerType()), StructField("distance_km", DoubleType()),
    StructField("courier_id", StringType()), StructField("event_time", TimestampType()),
])
restaurants = (spark.read.schema("restaurant_id STRING, name STRING, city STRING, category STRING, "
                                 "latitude DOUBLE, longitude DOUBLE")
               .json(f"{LAKE}/bronze/referentiel/restaurants.json")
               .select("restaurant_id", F.col("name").alias("restaurant_name"), "category"))

def build_clean():
    raw = (spark.readStream.format("kafka")
           .option("kafka.bootstrap.servers", "kafka:9092")
           .option("subscribe", "orders")
           .option("startingOffsets", "latest")      # ignoré au redémarrage : le checkpoint fait foi
           .option("failOnDataLoss", "false")
           .load())
    orders = (raw.select(F.from_json(F.col("value").cast("string"), orders_schema,
                                     {"timestampFormat": "yyyy-MM-dd'T'HH:mm:ssX"}).alias("o"))
                 .select("o.*")
                 .filter(F.col("order_id").isNotNull() & F.col("event_time").isNotNull()))
    return (orders
            .withWatermark("event_time", "10 minutes")
            .dropDuplicatesWithinWatermark(["order_id", "status"])     # 1. dédoublonnage
            .join(F.broadcast(restaurants), "restaurant_id", "left")    # 2. jointure stream-static
            .withColumn("distance_band",
                        F.when(F.col("distance_km") < 2, "court")
                         .when(F.col("distance_km") <= 5, "moyen")
                         .otherwise("long"))
            .withColumn("date", F.to_date("event_time")))

clean = build_clean()
clean.printSchema()

root
 |-- restaurant_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount_mad: double (nullable = true)
 |-- items_count: integer (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- restaurant_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- distance_band: string (nullable = false)
 |-- date: date (nullable = true)



In [17]:
def start_silver():
    return (build_clean().writeStream
            .queryName("p6_silver")
            .format("parquet")
            .option("path", SILVER)
            .option("checkpointLocation", f"{CKPT}/p6_silver")
            .partitionBy("city", "date")
            .outputMode("append")
            .trigger(processingTime="30 seconds")
            .start())

q_silver = start_silver()

26/11/15 19:06:07 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/15 19:06:30 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

In [4]:
def upsert_gold(batch_df, batch_id):
    s = batch_df.sparkSession
    kpi = batch_df.select(F.col("window.start").alias("window_start"),
                          F.col("window.end").alias("window_end"),
                          "city", "nb_commandes", "ca_mad", "nb_evenements")
    if kpi.isEmpty():
        return
    if not s._jvm.io.delta.tables.DeltaTable.isDeltaTable(s._jsparkSession, GOLD):
        kpi.write.format("delta").mode("overwrite").save(GOLD)
        return
    kpi.createOrReplaceTempView("kpi_updates")
    s.sql(f"""
        MERGE INTO delta.`{GOLD}` t
        USING kpi_updates u
        ON t.window_start = u.window_start AND t.city = u.city
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

def start_gold():
    tumbling = (build_clean()
                .groupBy(F.window("event_time", "10 minutes"), "city")
                .agg(F.count(F.when(F.col("status") == "CREATED", 1)).alias("nb_commandes"),
                     F.round(F.sum(F.when(F.col("status") == "DELIVERED", F.col("amount_mad"))), 2).alias("ca_mad"),
                     F.count("*").alias("nb_evenements")))
    return (tumbling.writeStream
            .queryName("p6_gold")
            .foreachBatch(upsert_gold)
            .option("checkpointLocation", f"{CKPT}/p6_gold")
            .outputMode("update")
            .trigger(processingTime="30 seconds")
            .start())

q_gold = start_gold()

26/11/15 18:56:37 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/15 18:56:37 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/11/15 18:57:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

In [5]:
sliding = (clean
           .groupBy(F.window("event_time", "10 minutes", "5 minutes"), "city")
           .agg(F.count(F.when(F.col("status") == "CREATED", 1)).alias("creees"),
                F.count(F.when(F.col("status") == "CANCELLED", 1)).alias("annulees"))
           .withColumn("taux_annulation_pct",
                       F.round(100 * F.col("annulees") / F.greatest(F.col("creees"), F.lit(1)), 2)))

q_sliding = (sliding.writeStream.queryName("p6_sliding")
             .format("console").outputMode("update")
             .option("truncate", False).option("numRows", 20)
             .trigger(processingTime="30 seconds").start())

26/11/15 18:58:19 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-8674d864-7844-4055-bbb9-1824cb647e75. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/15 18:58:19 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/15 18:58:19 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+------------------------------------------+---------+------+--------+-------------------+
|window                                    |city     |creees|annulees|taux_annulation_pct|
+------------------------------------------+---------+------+--------+-------------------+
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Marrakech|0     |0       |0.0                |
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Rabat    |0     |0       |0.0                |
|{2026-11-15 18:55:00, 2026-11-15 19:05:00}|Marrakech|0     |0       |0.0                |
|{2026-11-15 18:55:00, 2026-11-15 19:05:00}|Rabat    |0     |0       |0.0                |
+------------------------------------------+---------+------+--------+-------------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+----------+------+--------+-------------------+
|window                                    |city      |creees|annulees|taux_annulation_pct|
+------------------------------------------+----------+------+--------+-------------------+
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Tanger    |8     |1       |12.5               |
|{2026-11-15 18:45:00, 2026-11-15 18:55:00}|Casablanca|0     |1       |100.0              |
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Marrakech |12    |1       |8.33               |
|{2026-11-15 18:45:00, 2026-11-15 18:55:00}|Marrakech |0     |0       |0.0                |
|{2026-11-15 18:40:00, 2026-11-15 18:50:00}|Casablanca|0     |1       |100.0              |
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Rabat     |11    |2       |18.18              |
|{2026-11-15 18:45:00, 2026-11-15 18:55:00}|Rabat     |0     |0       |0.0 

In [21]:
sessions = (clean
            .groupBy(F.session_window("event_time", "5 minutes"), "courier_id", "city")
            .agg(F.count("*").alias("nb_evenements"))
            .withColumn("debut", F.col("session_window.start"))
            .withColumn("fin", F.col("session_window.end"))
            .withColumn("duree_min", F.round((F.unix_timestamp("fin") - F.unix_timestamp("debut")) / 60, 1))
            .drop("session_window")
            .orderBy(F.desc("duree_min")))

q_sessions = (sessions.writeStream.queryName("p6_sessions")
              .format("console").outputMode("complete")
              .option("truncate", False).option("numRows", 15)
              .trigger(processingTime="30 seconds").start())

26/11/15 19:13:44 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-76f84c0d-734b-4db5-9eb7-2e7b2e88d5d6. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/15 19:13:44 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/15 19:13:44 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+----------+------+-------------+-------------------+-------------------+---------+
|courier_id|city  |nb_evenements|debut              |fin                |duree_min|
+----------+------+-------------+-------------------+-------------------+---------+
|L-RAB-02  |Rabat |1            |2026-11-15 19:13:44|2026-11-15 19:18:44|5.0      |
|L-TAN-09  |Tanger|1            |2026-11-15 19:13:44|2026-11-15 19:18:44|5.0      |
+----------+------+-------------+-------------------+-------------------+---------+



-------------------------------------------
Batch: 1
-------------------------------------------
+----------+----------+-------------+-------------------+-------------------+---------+
|courier_id|city      |nb_evenements|debut              |fin                |duree_min|
+----------+----------+-------------+-------------------+-------------------+---------+
|L-RAB-10  |Rabat     |4            |2026-11-15 19:08:48|2026-11-15 19:18:56|10.1     |
|L-TAN-03  |Tanger    |6            |2026-11-15 19:10:46|2026-11-15 19:18:55|8.2      |
|L-CAS-09  |Casablanca|8            |2026-11-15 19:11:42|2026-11-15 19:18:54|7.2      |
|L-TAN-02  |Tanger    |3            |2026-11-15 19:12:37|2026-11-15 19:18:53|6.3      |
|L-RAB-01  |Rabat     |6            |2026-11-15 19:13:44|2026-11-15 19:18:59|5.3      |
|L-MAR-14  |Marrakech |5            |2026-11-15 19:13:48|2026-11-15 19:18:59|5.2      |
|L-RAB-13  |Rabat     |3            |2026-11-15 19:13:47|2026-11-15 19:18:59|5.2      |
|L-RAB-07  |Rabat     |

In [7]:
gold = spark.read.format("delta").load(GOLD)
gold.orderBy(F.desc("window_start"), "city").show(12, truncate=False)
spark.sql(f"DESCRIBE HISTORY delta.`{GOLD}`") \
     .select("version", "timestamp", "operation", "operationMetrics").show(10, truncate=60)

+-------------------+-------------------+----------+------------+--------+-------------+
|window_start       |window_end         |city      |nb_commandes|ca_mad  |nb_evenements|
+-------------------+-------------------+----------+------------+--------+-------------+
|2026-11-15 18:50:00|2026-11-15 19:00:00|Casablanca|414         |84297.77|1156         |
|2026-11-15 18:50:00|2026-11-15 19:00:00|Marrakech |213         |38011.93|562          |
|2026-11-15 18:50:00|2026-11-15 19:00:00|Rabat     |262         |49933.72|708          |
|2026-11-15 18:50:00|2026-11-15 19:00:00|Tanger    |163         |30716.74|429          |
|2026-11-15 18:40:00|2026-11-15 18:50:00|Casablanca|3           |2679.67 |23           |
|2026-11-15 18:40:00|2026-11-15 18:50:00|Marrakech |4           |1202.37 |10           |
|2026-11-15 18:40:00|2026-11-15 18:50:00|Rabat     |1           |328.99  |4            |
|2026-11-15 18:40:00|2026-11-15 18:50:00|Tanger    |0           |723.29  |4            |
+-------------------+

+-------+-------------------+---------+------------------------------------------------------------+
|version|          timestamp|operation|                                            operationMetrics|
+-------+-------------------+---------+------------------------------------------------------------+
|      5|2026-11-15 18:59:09|    MERGE|{numTargetRowsCopied -> 1, numTargetRowsDeleted -> 0, num...|
|      4|2026-11-15 18:58:39|    MERGE|{numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, num...|
|      3|2026-11-15 18:58:08|    MERGE|{numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, num...|
|      2|2026-11-15 18:57:38|    MERGE|{numTargetRowsCopied -> 1, numTargetRowsDeleted -> 0, num...|
|      1|2026-11-15 18:57:17|    MERGE|{numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, num...|
|      0|2026-11-15 18:56:53|    WRITE| {numFiles -> 2, numOutputRows -> 1, numOutputBytes -> 2598}|
+-------+-------------------+---------+----------------------------------------------------

-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+----------+------+--------+-------------------+
|window                                    |city      |creees|annulees|taux_annulation_pct|
+------------------------------------------+----------+------+--------+-------------------+
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Tanger    |73    |9       |12.33              |
|{2026-11-15 18:45:00, 2026-11-15 18:55:00}|Casablanca|4     |1       |25.0               |
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Marrakech |105   |11      |10.48              |
|{2026-11-15 18:45:00, 2026-11-15 18:55:00}|Marrakech |1     |0       |0.0                |
|{2026-11-15 18:40:00, 2026-11-15 18:50:00}|Rabat     |0     |0       |0.0                |
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Rabat     |147   |12      |8.16               |
|{2026-11-15 18:45:00, 2026-11-15 18:55:00}|Rabat     |2     |0       |0.0 

-------------------------------------------
Batch: 4
-------------------------------------------
+------------------------------------------+----------+------+--------+-------------------+
|window                                    |city      |creees|annulees|taux_annulation_pct|
+------------------------------------------+----------+------+--------+-------------------+
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Tanger    |101   |13      |12.87              |
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Marrakech |146   |15      |10.27              |
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Rabat     |201   |14      |6.97               |
|{2026-11-15 18:45:00, 2026-11-15 18:55:00}|Rabat     |3     |0       |0.0                |
|{2026-11-15 18:55:00, 2026-11-15 19:05:00}|Casablanca|298   |34      |11.41              |
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Casablanca|302   |34      |11.26              |
|{2026-11-15 18:55:00, 2026-11-15 19:05:00}|Rabat     |198   |14      |7.07

In [8]:
gold = spark.read.format("delta").load(GOLD)
gold.orderBy(F.desc("window_start"), "city").show(12, truncate=False)
spark.sql(f"DESCRIBE HISTORY delta.`{GOLD}`") \
     .select("version", "timestamp", "operation", "operationMetrics").show(10, truncate=60)

+-------------------+-------------------+----------+------------+---------+-------------+
|window_start       |window_end         |city      |nb_commandes|ca_mad   |nb_evenements|
+-------------------+-------------------+----------+------------+---------+-------------+
|2026-11-15 18:50:00|2026-11-15 19:00:00|Casablanca|594         |120213.34|1648         |
|2026-11-15 18:50:00|2026-11-15 19:00:00|Marrakech |298         |52419.75 |791          |
|2026-11-15 18:50:00|2026-11-15 19:00:00|Rabat     |376         |69089.2  |1001         |
|2026-11-15 18:50:00|2026-11-15 19:00:00|Tanger    |225         |49510.66 |629          |
|2026-11-15 18:40:00|2026-11-15 18:50:00|Casablanca|3           |2679.67  |23           |
|2026-11-15 18:40:00|2026-11-15 18:50:00|Marrakech |4           |1202.37  |10           |
|2026-11-15 18:40:00|2026-11-15 18:50:00|Rabat     |1           |328.99   |5            |
|2026-11-15 18:40:00|2026-11-15 18:50:00|Tanger    |0           |859.17   |5            |
+---------

[Stage 258:=> (4 + 2) / 8][Stage 260:>  (2 + 2) / 8][Stage 263:>  (0 + 0) / 8]

In [9]:
import boto3, json
s3 = boto3.client("s3", endpoint_url="http://minio:9000",
                  aws_access_key_id="minioadmin", aws_secret_access_key="minioadmin")

def dernier_offset_checkpoint(nom):
    keys = [o["Key"] for o in s3.list_objects_v2(Bucket="lake", Prefix=f"checkpoints/{nom}/offsets/")
            .get("Contents", []) if o["Key"].split("/")[-1].isdigit()]
    last = max(keys, key=lambda k: int(k.split("/")[-1]))
    contenu = s3.get_object(Bucket="lake", Key=last)["Body"].read().decode()
    return last, contenu

def compte_silver():
    df = spark.read.parquet(SILVER)
    return df.count(), df.select("order_id", "status").distinct().count()

time.sleep(5)
print("Silver AVANT arrêt  (lignes, couples distincts) :", compte_silver())
avant = q_silver.lastProgress["sources"][0]["endOffset"]
print("endOffset du dernier batch :", avant)
k, c = dernier_offset_checkpoint("p6_silver")
print(f"\nContenu de {k} :\n{c}")

q_silver.stop(); q_gold.stop()
t_stop = time.time()
print("\nRequêtes Silver et Gold arrêtées. Le producteur continue d'écrire dans Kafka…")

-------------------------------------------
Batch: 5
-------------------------------------------
+------------------------------------------+----------+------+--------+-------------------+
|window                                    |city      |creees|annulees|taux_annulation_pct|
+------------------------------------------+----------+------+--------+-------------------+
|{2026-11-15 19:00:00, 2026-11-15 19:10:00}|Marrakech |47    |5       |10.64              |
|{2026-11-15 18:45:00, 2026-11-15 18:55:00}|Casablanca|6     |1       |16.67              |
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Tanger    |102   |13      |12.75              |
|{2026-11-15 19:00:00, 2026-11-15 19:10:00}|Rabat     |53    |15      |28.3               |
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Marrakech |146   |15      |10.27              |
|{2026-11-15 19:00:00, 2026-11-15 19:10:00}|Tanger    |31    |3       |9.68               |
|{2026-11-15 18:45:00, 2026-11-15 18:55:00}|Marrakech |1     |0       |0.0 

[Stage 285:=============================================>          (9 + 2) / 11]

Silver AVANT arrêt  (lignes, couples distincts) : (5983, 5983)
endOffset du dernier batch : {'orders': {'2': 15382, '1': 15447, '0': 15159}}

Contenu de checkpoints/p6_silver/offsets/10 :
v1
{"batchWatermarkMs":1794768599000,"batchTimestampMs":1794769230030,"conf":{"spark.sql.streaming.stateStore.providerClass":"org.apache.spark.sql.execution.streaming.state.HDFSBackedStateStoreProvider","spark.sql.streaming.join.stateFormatVersion":"2","spark.sql.streaming.stateStore.compression.codec":"lz4","spark.sql.streaming.stateStore.rocksdb.formatVersion":"5","spark.sql.streaming.statefulOperator.useStrictDistribution":"true","spark.sql.streaming.flatMapGroupsWithState.stateFormatVersion":"2","spark.sql.streaming.multipleWatermarkPolicy":"min","spark.sql.streaming.aggregation.stateFormatVersion":"2","spark.sql.shuffle.partitions":"8"}}
{"orders":{"2":15382,"1":15447,"0":15159}}

Requêtes Silver et Gold arrêtées. Le producteur continue d'écrire dans Kafka…


-------------------------------------------
Batch: 6
-------------------------------------------
+------------------------------------------+----------+------+--------+-------------------+
|window                                    |city      |creees|annulees|taux_annulation_pct|
+------------------------------------------+----------+------+--------+-------------------+
|{2026-11-15 19:00:00, 2026-11-15 19:10:00}|Marrakech |95    |15      |15.79              |
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Tanger    |103   |13      |12.62              |
|{2026-11-15 18:45:00, 2026-11-15 18:55:00}|Casablanca|9     |1       |11.11              |
|{2026-11-15 19:00:00, 2026-11-15 19:10:00}|Rabat     |116   |17      |14.66              |
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Marrakech |147   |15      |10.2               |
|{2026-11-15 19:00:00, 2026-11-15 19:10:00}|Tanger    |59    |6       |10.17              |
|{2026-11-15 18:45:00, 2026-11-15 18:55:00}|Marrakech |1     |0       |0.0 

In [10]:
time.sleep(max(0, 120 - (time.time() - t_stop)))

q_silver = start_silver()
q_gold   = start_gold()

while not q_silver.recentProgress:      # attendre le 1er batch après la reprise
    time.sleep(5)
p = q_silver.recentProgress[0]
print("1er batch après reprise : batchId", p["batchId"])
print("  startOffset :", p["sources"][0]["startOffset"])
print("  endOffset   :", p["sources"][0]["endOffset"])
print("  lignes lues :", p["numInputRows"], "(≈ 2 min × 20 évts/s accumulés pendant l'arrêt)")
print("\nstartOffset == endOffset d'avant l'arrêt :",
      json.loads(p["sources"][0]["startOffset"]) == json.loads(avant)
      if isinstance(p["sources"][0]["startOffset"], str) else p["sources"][0]["startOffset"] == avant)

time.sleep(40)
print("\nSilver APRÈS reprise (lignes, couples distincts) :", compte_silver())

-------------------------------------------
Batch: 9
-------------------------------------------
+------------------------------------------+----------+------+--------+-------------------+
|window                                    |city      |creees|annulees|taux_annulation_pct|
+------------------------------------------+----------+------+--------+-------------------+
|{2026-11-15 19:00:00, 2026-11-15 19:10:00}|Marrakech |229   |34      |14.85              |
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Tanger    |106   |13      |12.26              |
|{2026-11-15 18:45:00, 2026-11-15 18:55:00}|Casablanca|11    |1       |9.09               |
|{2026-11-15 19:00:00, 2026-11-15 19:10:00}|Rabat     |266   |25      |9.4                |
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Marrakech |153   |15      |9.8                |
|{2026-11-15 19:00:00, 2026-11-15 19:10:00}|Tanger    |181   |13      |7.18               |
|{2026-11-15 18:45:00, 2026-11-15 18:55:00}|Marrakech |3     |0       |0.0 

26/11/15 19:02:42 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/15 19:02:43 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/11/15 19:02:43 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/15 19:02:43 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
[Stage 322:====>                                                   (4 + 4) / 50]

1er batch après reprise : batchId 11
  startOffset : {'orders': {'2': 15382, '1': 15447, '0': 15159}}
  endOffset   : {'orders': {'2': 16330, '1': 16385, '0': 16059}}
  lignes lues : 2786 (≈ 2 min × 20 évts/s accumulés pendant l'arrêt)

startOffset == endOffset d'avant l'arrêt : True


-------------------------------------------
Batch: 10
-------------------------------------------
+------------------------------------------+----------+------+--------+-------------------+
|window                                    |city      |creees|annulees|taux_annulation_pct|
+------------------------------------------+----------+------+--------+-------------------+
|{2026-11-15 19:00:00, 2026-11-15 19:10:00}|Marrakech |278   |39      |14.03              |
|{2026-11-15 18:45:00, 2026-11-15 18:55:00}|Casablanca|12    |1       |8.33               |
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Tanger    |108   |13      |12.04              |
|{2026-11-15 19:00:00, 2026-11-15 19:10:00}|Rabat     |315   |31      |9.84               |
|{2026-11-15 18:50:00, 2026-11-15 19:00:00}|Marrakech |154   |15      |9.74               |
|{2026-11-15 19:00:00, 2026-11-15 19:10:00}|Tanger    |209   |17      |8.13               |
|{2026-11-15 18:45:00, 2026-11-15 18:55:00}|Marrakech |4     |0       |0.0


Silver APRÈS reprise (lignes, couples distincts) : (9000, 9000)


In [11]:
for q in spark.streams.active:
    q.stop()
spark.stop()

26/11/15 19:03:33 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 11, writer: ConsoleWriter[numRows=20, truncate=false]] is aborting.
26/11/15 19:03:33 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 11, writer: ConsoleWriter[numRows=20, truncate=false]] aborted.
26/11/15 19:03:33 ERROR FileFormatWriter: Aborting job f29cdacb-0114-4764-8d5c-590ce96dab76.
java.lang.InterruptedException
	at java.base/java.util.concurrent.locks.AbstractQueuedSynchronizer.doAcquireSharedInterruptibly(Unknown Source)
	at java.base/java.util.concurrent.locks.AbstractQueuedSynchronizer.acquireSharedInterruptibly(Unknown Source)
	at scala.concurrent.impl.Promise$DefaultPromise.tryAwait(Promise.scala:242)
	at scala.concurrent.impl.Promise$DefaultPromise.ready(Promise.scala:258)
	at scala.concurrent.impl.Promise$DefaultPromise.ready(Promise.scala:187)
	at org.apache.spark.util.ThreadUtils$.awaitReady(ThreadUtils.scala:342)
	at org.apache.spark.sched